In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/njents_19may22.csv")
df.head(3)

,id,uuid,observed_on_string,observed_on,time_observed_at,time_zone,user_id,user_login,user_name,created_at,...,geoprivacy,taxon_geoprivacy,coordinates_obscured,positioning_method,positioning_device,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id
0,117696300,880aaa47-a4bc-4d05-8ca6-604ac45468f1,2022-05-19 02:20:33-04:00,2022-05-19,2022-05-19 06:20:33 UTC,Eastern Time (US & Canada),5686069,parroting,🐦,2022-05-19 06:24:37 UTC,...,NaN,NaN,False,NaN,NaN,Chestnut Carpenter Ant,Camponotus castaneus,Chestnut Carpenter Ant,Insecta,129272
1,117706028,cff715de-280f-4a3e-b303-7295e0f2a7d2,2022-05-19 05:35:10-04:00,2022-05-19,2022-05-19 09:35:10 UTC,Eastern Time (US & Canada),5686069,parroting,🐦,2022-05-19 09:36:08 UTC,...,NaN,NaN,False,NaN,NaN,Eastern Tent Caterpillar Moth,Malacosoma americana,Eastern Tent Caterpillar Moth,Insecta,211100
2,117722066,cbdf6d23-dc84-4dd7-8a1b-8b45899e4633,2022-05-19 09:16:51-04:00,2022-05-19,2022-05-19 13:16:51 UTC,Eastern Time (US & Canada),2225821,lizzkavanagh,NaN,2022-05-19 13:18:23 UTC,...,NaN,open,False,NaN,NaN,Polyphemus Moth,Antheraea polyphemus,Polyphemus Moth,Insecta,47919


In [2]:
# cleaning
import datetime as dt

df = df[df['time_observed_at'].notna()]
df["time_observed_at"] = pd.to_datetime(df["time_observed_at"], errors="coerce")
df['time_observed_at'] = df['time_observed_at'].dt.tz_convert('America/New_York')
df['time_observed_at'] = df['time_observed_at'].dt.tz_localize(None)

In [3]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import time as time

# Initialize using our freshly recovered database file
cache_session = requests_cache.CachedSession(
    ".cachup",
    backend="sqlite",
    use_cache=True,
    expire_after=-1,
    timeout=20,  # Forces the database to wait and retry instead of immediately locking up
)

retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [4]:
clean_df = df.dropna(
    subset=["latitude", "longitude", "time_observed_at"]
).copy()
clean_df = clean_df.head(200000)

clean_df["date_str"] = clean_df["time_observed_at"].dt.strftime("%Y-%m-%d")
clean_df["obs_hour"] = clean_df["time_observed_at"].dt.hour

In [5]:
chunk_size = 50
chunks = [
    clean_df.iloc[i : i + chunk_size]
    for i in range(0, len(clean_df), chunk_size)
]

all_results = []
url = "https://archive-api.open-meteo.com/v1/archive"

In [6]:
for idx, chunk in enumerate(chunks):
    params = {
        "latitude": chunk["latitude"].tolist(),
        "longitude": chunk["longitude"].tolist(),
        "start_date": chunk["date_str"].tolist(),
        "end_date": chunk["date_str"].tolist(),
        "hourly": ["temperature_2m", "relative_humidity_2m", "weather_code"],
        "timezone": "auto"
    }

    try:
        # Standard API fetch without retry wrappers
        responses = openmeteo.weather_api(url, params=params)

        matched_temps = []
        matched_humidities = []
        matched_codes = []

        # Parse the 1,000 timelines instantly in local memory
        for response, target_hour in zip(responses, chunk["obs_hour"]):
            hourly = response.Hourly()
            hourly_temps = hourly.Variables(0).ValuesAsNumpy()
            hourly_hums = hourly.Variables(1).ValuesAsNumpy()
            hourly_codes = hourly.Variables(2).ValuesAsNumpy()

            # Index directly into the specific target observation hour (0-23)
            matched_temps.append(hourly_temps[target_hour])
            matched_humidities.append(hourly_hums[target_hour])
            matched_codes.append(int(hourly_codes[target_hour]))

        # Save extracted weather vectors into the current chunk frame
        chunk = chunk.copy()
        chunk["hourly_temp_C"] = matched_temps
        chunk["hourly_humidity_percent"] = matched_humidities
        chunk["weather_code"] = matched_codes
        all_results.append(chunk)

    except Exception as e:
        print(f"Error processing chunk {idx}: {e}")
        # Append empty values so the rows aren't completely lost if an anomaly occurs
        chunk = chunk.copy()
        chunk["hourly_temp_C"] = np.nan
        chunk["hourly_humidity_percent"] = np.nan
        chunk["weather_code"] = np.nan
        all_results.append(chunk)
    # Print a status update every 10 chunks (500 rows)
    if (idx + 1) % 10 == 0 or (idx + 1) == len(chunks):
        print(f"Progress: Completed {idx + 1}/{len(chunks)} chunks...")
        #if idx > 1400:
        #    time.sleep(60)


# --- STEP 5: COMPILING & TRANSLATING TEXT DESCRIPTIONS ---
print("Stitching datasets back together...")
final_df = pd.concat(all_results, ignore_index=True)

Progress: Completed 10/3983 chunks...
Progress: Completed 20/3983 chunks...
Progress: Completed 30/3983 chunks...
Progress: Completed 40/3983 chunks...
Progress: Completed 50/3983 chunks...
Progress: Completed 60/3983 chunks...
Progress: Completed 70/3983 chunks...
Progress: Completed 80/3983 chunks...
Progress: Completed 90/3983 chunks...
Progress: Completed 100/3983 chunks...
Progress: Completed 110/3983 chunks...
Progress: Completed 120/3983 chunks...
Progress: Completed 130/3983 chunks...
Progress: Completed 140/3983 chunks...
Progress: Completed 150/3983 chunks...
Progress: Completed 160/3983 chunks...
Progress: Completed 170/3983 chunks...
Progress: Completed 180/3983 chunks...
Progress: Completed 190/3983 chunks...
Progress: Completed 200/3983 chunks...
Progress: Completed 210/3983 chunks...
Progress: Completed 220/3983 chunks...
Progress: Completed 230/3983 chunks...
Progress: Completed 240/3983 chunks...
Progress: Completed 250/3983 chunks...
Progress: Completed 260/3983 chunk

KeyboardInterrupt: 

In [7]:
if all_results:
    # Combine the 150 finished chunks
    partial_df = pd.concat(all_results, ignore_index=True)

    # Apply the weather label translations to the finished rows
    wmo_mapping = {
        0: "Sunny",
        1: "Sunny",
        2: "Partly Cloudy",
        3: "Cloudy",
        45: "Foggy",
        48: "Foggy",
        51: "Drizzle",
        53: "Drizzle",
        55: "Light Rain",
        56: "Freezing Drizzle",
        57: "Freezing Drizzle",
        61: "Light Rain",
        63: "Rain",
        65: "Rain",
        66: "Freezing Rain",
        67: "Freezing Rain",
        71: "Light Snow",
        73: "Snow",
        75: "Snow",
        77: "Snow",
        80: "Rain",
        81: "Rain",
        82: "Violent Rain",
        85: "Light Snow",
        86: "Snow",
        95: "Thunderstorm",
        96: "Hailstorm",
        99: "Hailstorm"
    }
    partial_df["weather_condition"] = partial_df["weather_code"].map(
        lambda x: wmo_mapping.get(x, "Cloudy/Variable")
        if pd.notna(x)
        else np.nan
    )

In [8]:
# partial_df.groupby("weather_condition")["common_name"].value_counts().groupby(level=0).head(5)
partial_df["day_of_year"] = pd.to_datetime(partial_df["time_observed_at"]).dt.dayofyear
display(partial_df.shape)
species_counts = partial_df["common_name"].value_counts()

min_observations = 75
viable_species = species_counts[species_counts >= min_observations].index

df_viable = partial_df.copy()
df_viable["target_species"] = df_viable["common_name"].apply(
    lambda x: x if x in viable_species else "Other Rare Insect"
)

df_viable = df_viable[df_viable["target_species"] != "Other Rare Insect"]

print(f"Training model on {len(viable_species)} distinct insect classes!")
print(
    f"The rarest included species is {viable_species[-1]}."
)



(77350, 47)

Training model on 205 distinct insect classes!
The rarest included species is Banded Tussock Moth.


In [9]:
import sklearn as sk
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_weather_encoded = pd.get_dummies(df_viable["weather_condition"], prefix="weather")
weather_columns = df_weather_encoded.columns.tolist()
df_weather_encoded = df_weather_encoded.astype(int)

df_viable["day_sin"] = np.sin(2 * np.pi * df_viable["day_of_year"] / 365.25)
df_viable["day_cos"] = np.cos(2 * np.pi * df_viable["day_of_year"] / 365.25)

df_viable["hour_sin"] = np.sin(2 * np.pi * df_viable["obs_hour"] / 24.0)
df_viable["hour_cos"] = np.cos(2 * np.pi * df_viable["obs_hour"] / 24.0)

base_params = ["hourly_temp_C", "hourly_humidity_percent", "hour_sin", "hour_cos", "day_sin", "day_cos", "latitude", "longitude"]

params = base_params + weather_columns

X = pd.concat([df_viable[base_params], df_weather_encoded], axis=1)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_viable["target_species"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67, stratify=y
)

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=100, min_samples_leaf=5, max_features="sqrt", random_state=67, n_jobs=-1, oob_score=True
)

rf_model.fit(X_train, y_train)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",67
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.

In [11]:
train_acc = rf_model.score(X_train, y_train)
test_acc = rf_model.score(X_test, y_test)

print(f"train acc: {train_acc:.2%}")
print(f"test acc: {test_acc:.2%}")

train acc: 63.93%
test acc: 16.11%


In [12]:
def topNacc(n, model, x, y):
    probs = model.predict_proba(x)
    topNindex = np.argsort(probs, axis=1)[:, -n:]
    match = np.any(topNindex == y.reshape(-1, 1), axis=1)

    return np.mean(match)

print(topNacc(20, rf_model, X_test, y_test))

0.6377979960843027


In [13]:
day = 170
hour = 14
weather = "Sunny"

custom_input_data = {
    "hourly_temp_C": 9,
    "hourly_humidity_percent": 57.0,
    "hour_sin": np.sin(2 * np.pi * hour / 24.0),
    "hour_cos": np.cos(2 * np.pi * hour / 24.0),
    "day_sin": np.sin(2 * np.pi * day / 365.25),
    "day_cos": np.cos(2 * np.pi * day / 365.25),
    "latitude": 40.7128, 
    "longitude": -74.0060,
}

for col in weather_columns:
    if col == f"weather_{weather}":
        custom_input_data[col] = 1
    else:
        custom_input_data[col] = 0


user_input_df = pd.DataFrame([custom_input_data])


user_probabilities = rf_model.predict_proba(user_input_df)[0]

top_n = 20
sorted_species_indices = np.argsort(user_probabilities)[::-1][:top_n]

print(f"Most likely species' today!")

for rank, idx in enumerate(sorted_species_indices, 1):
    species_name = label_encoder.inverse_transform([idx])[0]
    probability = user_probabilities[idx]
    
    if probability > 0:
        print(f"{rank}. {species_name:<30} | Match: {probability:.2%}")

Most likely species' today!
1. Spotted Lanternfly             | Match: 15.77%
2. Asian Lady Beetle              | Match: 8.73%
3. Western Honey Bee              | Match: 5.86%
4. Common Eastern Bumble Bee      | Match: 3.83%
5. Seven-spotted Lady Beetle      | Match: 3.56%
6. European Paper Wasp            | Match: 2.86%
7. Red-banded Hairstreak          | Match: 2.56%
8. Brown-belted Bumble Bee        | Match: 2.44%
9. Eastern Tailed-Blue            | Match: 2.03%
10. Spongy Oak Apple Gall Wasp     | Match: 2.03%
11. Fourteen-spotted Lady Beetle   | Match: 1.90%
12. Bald-faced Hornet              | Match: 1.88%
13. Eastern Tiger Swallowtail      | Match: 1.73%
14. Red Milkweed Beetle            | Match: 1.69%
15. Two-spotted Bumble Bee         | Match: 1.57%
16. Grapevine Beetle               | Match: 1.48%
17. Eastern Carpenter Bee          | Match: 1.40%
18. Polyphemus Moth                | Match: 1.37%
19. Eastern Tent Caterpillar Moth  | Match: 1.32%
20. Broad-necked Root Borer   

In [15]:
import joblib
joblib.dump(rf_model, "species_rf_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(weather_columns, "weather_columns.pkl")

['weather_columns.pkl']